# Milestone 3 – Comparable Systems

| Required evidence | Section |
|---|---|
| One baseline RAG response | 2 |
| One hybrid response for the same question and evidence | 2 |
| Valid citations or a visible rejection outcome | 2–3 |
| A supported abstention example | 4 |

The logic is in `src/pipeline.py`, so this notebook only runs it. Both systems receive the **same** retrieval result, prompt, and threshold. The only difference is whether the LoRA adapter is loaded.

The same comparison from the terminal:

```bash
uv run python -m src.pipeline --question "What must support collect for a duplicate payment investigation?"
```

In [ ]:
import json
import os
from pathlib import Path

# Run from the project root so data/, src/, artifacts/ and reports/ resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.generation import make_generator
from src.pipeline import answer_question, validate_answer
from src.prompts import FALLBACK_ANSWER, evidence_text
from src.retrieval import Retriever, build_chunks
from src.training import TrainingConfig

print("Working directory:", Path.cwd())

## 1. One retriever, two generators

The index is built once. The baseline loads the unchanged base model. The hybrid loads the same pinned base model plus the saved adapter.

In [ ]:
retriever = Retriever()
retriever.index(build_chunks())
print(f"top_k={retriever.top_k}  min_score={retriever.min_score}")

adapter_path = TrainingConfig().adapter_path
systems = {
    "baseline-rag": make_generator(None),
    "hybrid-rag-lora": make_generator(adapter_path),
}
print("Adapter:", adapter_path)

In [ ]:
def compare(question):
    """Retrieve once, then answer with both systems from that same evidence."""

    retrieval = retriever.retrieve(question)
    print("Q:", question)
    print("Retrieved:", retrieval.source_ids or "(none above min_score)")
    print("Evidence passed to both models:")
    print("  " + (evidence_text(retrieval.hits).replace("\n", "\n  ") or "(none)"))
    answers = {}
    for name, generate in systems.items():
        answer = answer_question(name, retrieval, generate)
        answers[name] = answer
        print(f"\n[{name}] {answer.status.upper()}: {answer.reason}  ({answer.latency:.2f}s)")
        print("  citations:", answer.citations or "-")
        print("  raw:      ", answer.raw or "(not generated)")
        print("  served:   ", answer.text)
    return answers

## 2. Baseline vs hybrid on the same question and evidence

`served` is what the customer would see. A rejected answer is never served: the fallback reply is shown instead, and `raw` keeps the model output for inspection.

In [ ]:
answers = compare("What must support collect for a duplicate payment investigation?")

In the recorded run, the **baseline** found the right facts (invoice number, payment references) but wrote no citation, so it was **rejected**.
The **hybrid** cited `[refunds]` and `[response-style]`. Both were retrieved, so the answer was **answered**.
The hybrid's answer also copies an unrelated rule (the 24-hour refund window). So a valid citation shows format compliance, not correctness.

## 3. Citation validation rules

Citations are checked against the IDs that were actually retrieved for this question.

In [ ]:
retrieved_ids = ["shipping"]
examples = {
    "valid citation":       "Support cannot change the address after packing. [shipping]",
    "no citation":          "Support cannot change the address after packing.",
    "citation not retrieved": "A refund is available within 30 days. [refunds]",
    "model abstains":       FALLBACK_ANSWER,
}
for label, text in examples.items():
    status, reason, citations = validate_answer(text, retrieved_ids)
    print(f"{label:<24} -> {status:<9} ({reason})")

## 4. Abstention

An off-topic question scores below `min_score`. Neither system generates anything, and both return the fallback reply.

In [ ]:
abstain = compare("Where can I find next week's cafeteria menu?")
assert all(a.status == "abstained" for a in abstain.values())

### Limit: near-domain unsupported questions

This question is about the duplicate-payment topic, but the policies do not answer it. Retrieval scores it above the threshold, so abstaining is left to the model.

In [ ]:
near_domain = compare("How many days will my external bank take to resolve a chargeback for a duplicate payment?")

In the recorded run, the baseline correctly said that the context does not cover this, but it was rejected for having no citation.
The hybrid invented a timeline and cited `[refunds]`. Because `refunds` was retrieved, the validator **served** this unsupported claim.
This is the main failure analysed in `notebooks/04_benchmark.ipynb` and `reports/evaluation_report.md`.